# Benchmark: KTVIC + Vista Conversation

This notebook downloads the two active benchmarks and evaluates the Stage 1 / Stage 2 checkpoints. It also visualizes metrics and sample predictions.

In [ ]:
!pip -q install -U gdown datasets pillow pandas pyarrow matplotlib seaborn tqdm pyyaml loguru beautifulsoup4 requests openai python-dotenv kaggle

import json
import os
import random
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import torch
try:
    from google.colab import files, userdata
except Exception:
    files = None
    userdata = None

In [ ]:
REPO_URL = "https://github.com/duongtruongbinh/pretrain_vlm.git"
BRANCH = "heisenburger"
PROJECT_DIR = Path("/content/pretrain_vlm")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Project root:", Path.cwd())

## Configure Kaggle access for KTVIC

KTVIC is hosted on Kaggle. Provide `kaggle.json` either through Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) or upload the file manually.

In [ ]:
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(exist_ok=True)

username = None
key = None
if userdata is not None:
    try:
        username = userdata.get("KAGGLE_USERNAME")
        key = userdata.get("KAGGLE_KEY")
    except Exception:
        pass

if username and key:
    (KAGGLE_DIR / "kaggle.json").write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
elif not (KAGGLE_DIR / "kaggle.json").exists() and files is not None:
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        shutil.copy("kaggle.json", KAGGLE_DIR / "kaggle.json")

if (KAGGLE_DIR / "kaggle.json").exists():
    os.chmod(KAGGLE_DIR / "kaggle.json", 0o600)
    print("Kaggle credentials ready.")
else:
    print("Kaggle credentials not found. KTVIC download will fail until kaggle.json is provided.")

## Download benchmark data

In [ ]:
subprocess.run(["python", "scripts/download_benchmarks.py", "--output-root", "data/benchmarks"], check=True)

## Set checkpoint paths

On Colab, place checkpoints in Drive or upload them manually, then update these paths.

In [ ]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
STAGE1_CHECKPOINT = "outputs/stage_1_projector_coco_uit_test/checkpoint-7000"
STAGE2_CHECKPOINT = "outputs/stage_2_instruction_final/checkpoint-6000"
MAX_SAMPLES = 20  # Set to None for the full benchmark.

print("device:", DEVICE)
print("stage1 exists:", Path(STAGE1_CHECKPOINT).exists(), STAGE1_CHECKPOINT)
print("stage2 exists:", Path(STAGE2_CHECKPOINT).exists(), STAGE2_CHECKPOINT)

## Run KTVIC and Vista evaluation

In [ ]:
def run_eval(command):
    if MAX_SAMPLES is not None:
        command += ["--max-samples", str(MAX_SAMPLES)]
    print(" ".join(command))
    subprocess.run(command, check=True)

run_eval([
    "python", "scripts/evaluate.py", "ktvic",
    "--annotations", "data/benchmarks/ktvic/raw/ktvic_dataset/test_data.json",
    "--image-root", "data/benchmarks/ktvic/raw/ktvic_dataset/public-test-images",
    "--checkpoint", STAGE1_CHECKPOINT,
    "--output-dir", "outputs/benchmarks/ktvic",
    "--device", DEVICE,
])

run_eval([
    "python", "scripts/evaluate.py", "vista-conversation",
    "--annotations", "data/benchmarks/vista/data/vi_llava_conversation/validation-00000-of-00001.parquet",
    "--image-root", "data/benchmarks/vista/images/coco2017/val2017",
    "--checkpoint", STAGE2_CHECKPOINT,
    "--output-dir", "outputs/benchmarks/vista_conversation",
    "--device", DEVICE,
])

## Visualize metrics and predictions

In [ ]:
def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def plot_counts(title, counts):
    series = pd.Series(counts, dtype="int64")
    ax = series.plot(kind="bar", figsize=(7, 4), color="#4C78A8")
    ax.set_title(title)
    ax.set_ylabel("rows")
    ax.bar_label(ax.containers[0])
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def show_image_grid(records, text_getter, n=6):
    samples = [r for r in records if Path(str(r.get("image", ""))).exists()]
    if not samples:
        print("No local image samples found.")
        return
    samples = random.sample(samples, min(n, len(samples)))
    cols = min(3, len(samples))
    rows = (len(samples) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4.5 * rows))
    for i, record in enumerate(samples, start=1):
        image = Image.open(record["image"]).convert("RGB")
        ax = plt.subplot(rows, cols, i)
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(text_getter(record)[:120], fontsize=9)
    plt.tight_layout()
    plt.show()

ktvic_metrics = json.loads(Path("outputs/benchmarks/ktvic/metrics.json").read_text(encoding="utf-8"))
vista_metrics = json.loads(Path("outputs/benchmarks/vista_conversation/metrics.json").read_text(encoding="utf-8"))

display(pd.DataFrame([
    {"benchmark": "KTVIC", "metric": k, "value": v}
    for k, v in ktvic_metrics.items() if isinstance(v, (int, float)) and k != "count"
] + [
    {"benchmark": "Vista", "metric": k, "value": v}
    for k, v in vista_metrics.items() if isinstance(v, (int, float)) and k != "count"
]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series({k: v for k, v in ktvic_metrics.items() if isinstance(v, (int, float)) and k != "count"}).plot(kind="bar", ax=axes[0], title="KTVIC metrics")
pd.Series({k: v for k, v in vista_metrics.items() if isinstance(v, (int, float)) and k != "count"}).plot(kind="bar", ax=axes[1], title="Vista metrics")
plt.tight_layout()
plt.show()

ktvic_rows = read_jsonl("outputs/benchmarks/ktvic/predictions.jsonl", limit=100)
vista_rows = read_jsonl("outputs/benchmarks/vista_conversation/predictions.jsonl", limit=100)

print("KTVIC samples")
show_image_grid(ktvic_rows, lambda r: f"P: {r.get('prediction', '')}\nR: {(r.get('references') or [''])[0]}", n=6)

print("Vista samples")
show_image_grid(vista_rows, lambda r: f"Q: {r.get('question', '')[-80:]}\nP: {r.get('prediction', '')[:100]}", n=6)